# Public archival edition
Original academic source with outputs and execution metadata removed.
Licensed input data are excluded. See ../docs/EVIDENCE_AND_LIMITATIONS.md before interpreting calculations.
This notebook has not been independently rerun or certified as a valid trading backtest.

# FACTOR DATA AND ANALYSIS（BACKTEST）

## Implementing portfolio selection

### Import Library

In [ ]:
import pandas as pd

import numpy as np
from numpy import *
from numpy.linalg import multi_dot  

import matplotlib.pyplot as plt
from matplotlib.pyplot import rcParams 
rcParams['figure.figsize'] = 16, 8 




### Retrive-Data


In [ ]:
def read_data(filename, sheet_name):  # Read the data in excel sheet
    df = pd.read_excel(filename, sheet_name = sheet_name)
    df.set_index("Date", inplace=True)      # Indexed by the Date column
    
    df = df.astype('float64')               # Convert the data to float64 format
    return df

In [ ]:
# original data
## Stock price data
PRICE_FILENAME= './Data/Price_Data.xlsx'
PRICE_SHEETNAME= 'Weekly'

## Stock market capitalization data
MV_FILENAME= './Data/Market_Value.xlsx'
MV_SHEETNAME= 'Weekly'

In [ ]:
read_data(PRICE_FILENAME,PRICE_SHEETNAME)

In [ ]:
df=read_data(PRICE_FILENAME,PRICE_SHEETNAME)
df = df.iloc[:, 3:]
df

### Descriptive Statistics


In [ ]:
summary = df.describe().T

In [ ]:
summary

### Visualize Data

In [ ]:

# Visualize the data
fig = plt.figure(figsize=(16,8))
ax = plt.axes()

ax.set_title('Normalized Price Plot')
ax.plot(df[-252:]/df.iloc[-252] * 100)
ax.legend(df.columns, loc='upper left')
ax.grid(True)

### Calculate Returns

In [ ]:
# Calculate returns 
returns = df.pct_change().fillna(0)
returns

### Annualized Returns

In [ ]:
# Calculate annual returns
annual_returns = (returns.mean() * 52)
annual_returns

In [ ]:
# Visualize the data
fig = plt.figure()
ax =plt.axes()

ax.bar(annual_returns.index, annual_returns*100, color='royalblue', alpha=0.75)
ax.set_title('Annualized Returns (in %)');


### Calculate Volatility

In [ ]:
vols = returns.std()
vols

### Annualized Volatilities

In [ ]:
# Calculate annualized volatilities
annual_vols = vols*sqrt(52)
annual_vols


In [ ]:
# Visualize the data
fig = plt.figure()
ax = plt.axes()

ax.bar(annual_vols.index, annual_vols*100, color='orange', alpha=0.5)
ax.set_title('Annualized Volatility (in %)');


### Equal weighted portfolio


(Assume a portoflio composed of all five stocks with equal weighting. We will now calculate the portfolio statistics.)

In [ ]:
numofasset = 10 



In [ ]:
wts = numofasset * [1./numofasset]
wts = array(wts)[:,newaxis]
wts

In [ ]:
wts.T

In [ ]:
wts.shape

### Portfolio Return

In [ ]:
array(returns.mean() * 52)[:,newaxis] 

In [ ]:
array(returns.mean() * 52)[:,newaxis].shape 

In [ ]:
# Portfolio returns
wts.T @ array(returns.mean() * 52)[:,newaxis] 

### Portfolio Volatility

In [ ]:
# Covariance matrix
returns.cov() * 52

In [ ]:
# Portfolio variance
multi_dot([wts.T,returns.cov()*52,wts])

In [ ]:
# Portfolio volatility
sqrt(multi_dot([wts.T,returns.cov()*52,wts]))

### Portfolio Statistics

In [ ]:
def portfolio_stats(weights):
    
    weights = array(weights)[:,newaxis]
    port_rets = weights.T @ array(returns.mean() * 52)[:,newaxis]    
    port_vols = sqrt(multi_dot([weights.T, returns.cov() * 52, weights])) 
    
    return np.array([port_rets, port_vols, port_rets/port_vols]).flatten()

### Portfolio Simulation

Now, we will implement a Monte Carlo simulation to generate random portfolio weights on a larger scale and calculate the expected portfolio return, variance and sharpe ratio for every simulated allocation. We will then identify the portfolio with a highest return for per unit of risk.

In [ ]:
w = random.random(numofasset)[:, newaxis]
w

In [ ]:
w /= sum(w)
w

In [ ]:
w.shape, sum(w)

In [ ]:
w.flatten()

In [ ]:
# Initialize the lists
rets = []; vols = []; wts = []

# Simulate 10,000 portfolios
for i in range (10000):
    
    # Generate random weights
    weights = random.random(numofasset)[:, newaxis]
    
    # Set weights such that sum of weights equals 1
    weights /= sum(weights)
    
    # Portfolio statistics
    rets.append(weights.T @ array(returns.mean() * 52)[:, newaxis])        
    vols.append(sqrt(multi_dot([weights.T, returns.cov()*52, weights])))
    wts.append(weights.flatten())

# Record values     
port_rets = array(rets).flatten()
port_vols = array(vols).flatten()
port_wts = array(wts)


In [ ]:
port_rets

In [ ]:
port_vols

In [ ]:
port_wts

In [ ]:
port_rets.shape, port_vols.shape, port_wts.shape

In [ ]:
# Create a dataframe for analysis
mc_df = pd.DataFrame({'returns': port_rets,
                      'volatility': port_vols,
                      'sharpe_ratio': port_rets/port_vols,
                      'weights': list(port_wts)})
mc_df.head()


### Summury Statistics

In [ ]:
# Summary Statistics
mc_df.describe().T

### Maximum Sharp Ration Portfolio

In [ ]:
# Max sharpe ratio portfolio 
msrp = mc_df.iloc[mc_df['sharpe_ratio'].idxmax()]
msrp


In [ ]:

symbols = df.columns.tolist()
symbols

In [ ]:
# Max sharpe ratio portfolio weights
max_sharpe_port_wts = mc_df['weights'][mc_df['sharpe_ratio'].idxmax()]

# Allocation to achieve max sharpe ratio portfolio
dict(zip(symbols,np.around(max_sharpe_port_wts*100,2)))



### Visulize Simulated Portfolio

In [ ]:

# Visualize the simulated portfolio for risk and return
fig = plt.figure()
ax = plt.axes()

ax.set_title('Monte Carlo Simulated Allocation')

# Simulated portfolios
fig.colorbar(ax.scatter(port_vols, port_rets, c=port_rets / port_vols, 
                        marker='o', cmap='RdYlGn', edgecolors='black'), label='Sharpe Ratio') 

# Maximum sharpe ratio portfolio
ax.scatter(msrp['volatility'], msrp['returns'], c='red', marker='*', s = 300, label='Max Sharpe Ratio')

ax.set_xlabel('Expected Volatility')
ax.set_ylabel('Expected Return')
ax.grid(True)

## Introducing factors

### HML factor

**retrieve data**

In [ ]:
HML_FILENAME= './HML/MV-PV.xlsx'
HML_SHEETNAME= 'PB'


In [ ]:
df_HML=pd.read_excel(HML_FILENAME, sheet_name=HML_SHEETNAME)
df_HML

**distinguish between value stocks and incremental stocks**

In [ ]:
# calculate the inverse of the market-to-book ratio
df_HML = 1 / df_HML


In [ ]:
# find the median of the inverse of the market-to-book ratio
median_value = df_HML.iloc[0].median()

In [ ]:
# stocks are classified into value and incremental stocks based on the median
value_stocks = df_HML.columns[df_HML.iloc[0] > median_value].tolist()
incremental_stocks = df_HML.columns[df_HML.iloc[0] < median_value].tolist()

In [ ]:
# output result
print("价值股（Value Stocks）:", value_stocks)
print("增量股（Incremental Stocks）:", incremental_stocks)

In [ ]:
# Reorder returns by category
df_sorted = returns[value_stocks + incremental_stocks]
print(df_sorted.head())  # 显示排序后DataFrame的前几行

In [ ]:
# calculate the average return for value and incremental stocks in each row
value_stock_returns = df_sorted[value_stocks].mean(axis=1)
incremental_stock_returns = df_sorted[incremental_stocks].mean(axis=1)

# Calculate the difference between the return on value shares and the return on incremental shares
df_sorted['HML'] = value_stock_returns - incremental_stock_returns

df_sorted


### SMB factor

**retrieve data**

In [ ]:
SMB_FILENAME= './SMB/MV.xlsx'
SMB_SHEETNAME= 'MV'


In [ ]:
df_SMB=pd.read_excel(SMB_FILENAME, sheet_name=SMB_SHEETNAME)
df_SMB

In [ ]:
df_SMB = df_SMB.drop(df_SMB.columns[0], axis=1)
df_SMB

**distinguish between small-cap stocks and large-cap stocks**

In [ ]:
# find the median market-to-market value
median_value = df_SMB.iloc[0].median()

In [ ]:
# Stocks are classified into small and large cap stocks based on the median
large_mv= df_SMB.columns[df_SMB.iloc[0] > median_value].tolist()
small_mv= df_SMB.columns[df_SMB.iloc[0] < median_value].tolist()

In [ ]:
# output
print("large cap stocks（large_mv）:", large_mv)
print("small cap stocks（small_mv）:", small_mv)

In [ ]:
# reorder returns by category
df_MV = returns[large_mv + small_mv]
print(df_MV) 

In [ ]:
# calculate the average return of large-cap and small-cap stocks in each row
large_mv_returns = df_MV[large_mv].mean(axis=1)
small_mv_returns = df_MV[small_mv].mean(axis=1)

# calculate the average weekly return of the small-cap portfolio minus the average weekly return of the large-cap portfolio
df_MV['SMB'] = small_mv_returns - large_mv_returns

df_MV


In [ ]:
# the last column of df_MV is extracted
last_column = df_MV.iloc[:, -1]

# add this column to the end of df_sorted
df_sorted[last_column.name] = last_column

# assign the result to df
df = df_sorted

df

## Factor analysis

In [ ]:
# data exploration：
'''
check data for completeness and consistency. Make sure there are no missing values or inconsistent data points.
descriptive statistical analysis was performed, such as calculating mean, standard deviation, minimum and maximum values, etc.
visualize data, such as using a line graph to see trends in stock returns and factors over time.
'''

# 1. check data for completeness
print("check data for completeness:")
print(df.isnull().sum())

# 2. descriptive statistical analysis
print("\ndescriptive statistical analysis:")
print(df.describe())

# 3. Visualizing data - in the case of AAPL.O
plt.figure(figsize=(10, 6))
plt.plot(df.index, df['AAPL.O'], label='AAPL.O')
plt.title('AAPL.O Returns Over Time')
plt.xlabel('Date')
plt.ylabel('Returns')
plt.legend()
plt.show()



$ R_{it} - R_{ft} = \alpha_i + \beta_{i,MKT}(R_{Mt} - R_{ft}) + \beta_{i,SMB} \cdot SMB_t + \beta_{i,HML} \cdot HML_t + \epsilon_{it} $

Where:
- \( R_{it} \): Return of asset \( i \) at time \( t \)
- \( R_{ft} \): Risk-free rate at time \( t \)
- \( R_{Mt} - R_{ft} \): Market excess return (return of the market portfolio minus the risk-free rate)
- \( \alpha_i \): Intercept term for asset \( i \), representing asset-specific return
- \( \beta_{i,MKT} \): Sensitivity of asset \( i \) to market risk premium
- \( \beta_{i,SMB} \): Sensitivity of asset \( i \) to SMB (Small Minus Big) factor
- \( \beta_{i,HML} \): Sensitivity of asset \( i \) to HML (High Minus Low) factor
- \( SMB_t \): Difference in returns between small and big market capitalization stocks
- \( HML_t \): Difference in returns between high and low book-to-market ratio stocks
- \( \epsilon_{it} \): Error term


assume：risk free return : 3M US Treasury from pandas FRED dataset

In [ ]:
# retrieve risk free return

import pandas_datareader as pdr
from datetime import datetime

# define the start and end dates
start = datetime(2000, 1, 1)
end = datetime(2023, 1, 1)

# data on 3-month Treasury yields were obtained from FRED
try:
    df_risk_free_rate = pdr.get_data_fred('DGS3MO', start=start, end=end)
    # calculate the arithmetic average rate of return
    arithmetic_mean_return = df_risk_free_rate['DGS3MO'].mean()
    print(f"The arithmetic average yield on three-month Treasury bills is: {arithmetic_mean_return}%")
    Rf=arithmetic_mean_return
except Exception as e:
    print(f"Error: {e}")


In [ ]:
# regression

import statsmodels.api as sm

# A stock and factor are selected for regression analysis
# Here 'AAPL.O' is used as an example, and both 'HML' and 'SMB' are used as factors
X = df[['HML', 'SMB']]  # Independent variable (factor)
y = df['AAPL.O']        # Dependent variable (stock return)

# Constant terms are added so that the model includes the intercept
X = sm.add_constant(X)

# A linear regression model was constructed and fitted
model = sm.OLS(y, X).fit()

# output
model_summary = model.summary()
model_summary



In [ ]:
import seaborn as sns

# Extract the stock name
stock_names = df.columns[:10]  # Assume that the first 10 columns, except for the time index, are stock data

# Create an empty list to store the regression results
regression_results = []

# The regression analysis is performed for each stock
for stock in stock_names:
    y = df[stock]
    X = df[['HML', 'SMB']]
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    
    # Add the results to the list
    regression_results.append({
        'Stock': stock,
        'HML_coef': model.params['HML'],
        'SMB_coef': model.params['SMB'],
        'p_value_HML': model.pvalues['HML'],
        'p_value_SMB': model.pvalues['SMB'],
        'R_squared': model.rsquared
    })

# Convert the list to a DataFrame
regression_results_df = pd.DataFrame(regression_results)

# Visual regression coefficient
plt.figure(figsize=(14, 6))

# HML visualization of coefficients
plt.subplot(1, 2, 1)
sns.barplot(x='HML_coef', y='Stock', data=regression_results_df, palette='coolwarm')
plt.title('HML Coefficients for Each Stock')

# SMB visualization of coefficients
plt.subplot(1, 2, 2)
sns.barplot(x='SMB_coef', y='Stock', data=regression_results_df, palette='coolwarm')
plt.title('SMB Coefficients for Each Stock')

plt.tight_layout()
plt.show()

# A summary of the regression results is shown
regression_results_df



### Show profit and loss (P&L) rate of return

In [ ]:
df_index=read_data(PRICE_FILENAME,PRICE_SHEETNAME)
df_index = df_index.iloc[:,:1]
df_index

In [ ]:
# Calculate returns 
returns_index = df_index.pct_change().fillna(0)
df_index=returns_index
df_index

In [ ]:
# The last column of df_MV is extracted
last_column = df_index.iloc[:, -1]

# Add this column to the end of df_sorted
df[last_column.name] = last_column

# Assign the result to df
df_P= df

df_P

In [ ]:


# Calculate cumulative rate of return
df_P['Cumulative_HML_Return'] = (1 + df_P['HML']).cumprod() - 1
df_P['Cumulative_SMB_Return'] = (1 + df_P['SMB']).cumprod() - 1
df_P['Cumulative_Market_Index_Return'] = (1 + df_P['SPX.GI']).cumprod() - 1

# Visual cumulative return
plt.figure(figsize=(10, 6))
plt.plot(df_P['Cumulative_HML_Return'], label='HML Factor')
plt.plot(df_P['Cumulative_SMB_Return'], label='SMB Factor')
plt.plot(df_P['Cumulative_Market_Index_Return'], label='Market Index')
plt.title('Cumulative Returns Over Time')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.legend()
plt.show()



### performance analysis
Evaluating the performance of the factor relative to the market index may include indicators such as cumulative return, Sharpe ratio, maximum retracement, etc.
Display rolling Beta and changing Alpha: Beta measures the volatility of a stock or portfolio relative to the market as a whole; Alpha represents the expected excess return of the portfolio relative to the market benchmark. Rolling betas and alphas can reveal market sensitivity and portfolio performance over time.

#### Independent factor analysis

In [ ]:
# Measures such as cumulative return, Sharpe ratio and maximum retracement are calculated
# Calculate Sharpe ratio
HML_sharpe_ratio = (df_P['HML'].mean() - Rf) / df_P['HML'].std()
SMB_sharpe_ratio = (df_P['SMB'].mean() - Rf) / df_P['SMB'].std()
market_sharpe_ratio = (df_P['SPX.GI'].mean() - Rf) / df_P['SPX.GI'].std()

# Calculate the maximum retracement
def max_drawdown(cumulative_returns):
    peak = cumulative_returns.cummax()
    drawdown = (peak - cumulative_returns) / peak
    return drawdown.max()

HML_max_drawdown = max_drawdown(df_P['Cumulative_HML_Return'])
SMB_max_drawdown = max_drawdown(df_P['Cumulative_SMB_Return'])
market_max_drawdown = max_drawdown(df_P['Cumulative_Market_Index_Return'])


# Calculate the rolling Beta and Alpha
window = 60
factors = ['HML', 'SMB']
for factor in factors:
    rolling_beta = []
    rolling_alpha = []
    for i in range(len(df_P) - window + 1):
        window_df = df_P.iloc[i:i+window]
        X = sm.add_constant(window_df['SPX.GI'])
        model = sm.OLS(window_df[factor], X).fit()
        rolling_beta.append(model.params['SPX.GI'])
        rolling_alpha.append(model.params['const'])
    
    # Visualize the rolling Beta and Alpha
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.plot(df_P.index[window-1:], rolling_beta, label=f'Rolling Beta - {factor}')
    plt.title(f'Rolling Beta of {factor}')
    plt.xlabel('Date')
    plt.ylabel('Beta')

    plt.subplot(1, 2, 2)
    plt.plot(df_P.index[window-1:], rolling_alpha, label=f'Rolling Alpha - {factor}')
    plt.title(f'Rolling Alpha of {factor}')
    plt.xlabel('Date')
    plt.ylabel('Alpha')

    plt.tight_layout()
    plt.show()


#### Combined factor analysis

In [ ]:


# Create combined factors - such as simple averages
df_P['Combined_Factor'] = (df_P['HML'] + df_P['SMB']) / 2

# Measures such as cumulative return, Sharpe ratio and maximum retracement are calculated
# ...

# Calculate the rolling Beta and Alpha
window = 60
rolling_beta = []
rolling_alpha = []
for i in range(len(df_P) - window + 1):
    window_df = df_P.iloc[i:i+window]
    X = sm.add_constant(window_df['SPX.GI'])
    model = sm.OLS(window_df['Combined_Factor'], X).fit()
    rolling_beta.append(model.params['SPX.GI'])
    rolling_alpha.append(model.params['const'])

# Visualize the rolling Beta and Alpha
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(df_P.index[window-1:], rolling_beta, label='Rolling Beta - Combined Factor')
plt.title('Rolling Beta of Combined Factor')
plt.xlabel('Date')
plt.ylabel('Beta')

plt.subplot(1, 2, 2)
plt.plot(df_P.index[window-1:], rolling_alpha, label='Rolling Alpha - Combined Factor')
plt.title('Rolling Alpha of Combined Factor')
plt.xlabel('Date')
plt.ylabel('Alpha')

plt.tight_layout()
plt.show()
